<a href="https://colab.research.google.com/github/human530/MD.Piece/blob/claude%2Fautoresearch-feature-ynJbT/autoresearch_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoresearch on Colab
基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：** Runtime → Change runtime type → 選 GPU (T4 或更高)

> T4 (Turing) 也可以跑！本 notebook 會自動偵測 GPU 並 patch Flash Attention fallback。

In [ ]:
# 確認 GPU 並偵測架構
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"\n✅ GPU: {name} (compute capability: {cap[0]}.{cap[1]})")
    if cap[0] >= 8:
        print("   Ampere+ → Flash Attention 3 原生支援")
    else:
        print("   Pre-Ampere → 將自動套用 SDPA fallback patch")
else:
    print("❌ 未偵測到 GPU！請到 Runtime → Change runtime type → 選 GPU")

In [12]:
# 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [13]:
# Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

Cloning into 'autoresearch'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 188 (delta 0), reused 0 (delta 0), pack-reused 186 (from 2)
Receiving objects: 100% (188/188), 529.07 KiB | 12.02 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/autoresearch/autoresearch/autoresearch


In [ ]:
# 安裝依賴
!uv sync

# ── T4 Patch（直接嵌入，不需要從 GitHub 下載） ──
import re, torch
from pathlib import Path

train_path = Path("train.py")
if not train_path.exists():
    train_path = Path("autoresearch/train.py")

if train_path.exists():
    cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
    if cap[0] < 8:
        print(f"[t4_patch] Pre-Ampere GPU (sm_{cap[0]}{cap[1]}) — applying T4 patch...")
        text = train_path.read_text()
        original = text

        # Patch 1: Flash Attention 3 → SDPA fallback
        fa3_replacement = """import torch
_gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
_is_ampere_plus = _gpu_cap[0] >= 8

if _is_ampere_plus:
    import kernels
    fa3 = kernels.get_kernel("flash_attn3")
else:
    fa3 = None  # Will use SDPA fallback"""

        text = re.sub(
            r'import kernels\s*\nfa3\s*=\s*kernels\.get_kernel\(["\']flash_attn3["\']\)',
            fa3_replacement, text
        )

        # Patch 2: fa3 call → SDPA fallback
        text = re.sub(
            r'y = fa3\.flash_attn_func\(q, k, v, causal=True, window_size=window_size\)',
            """if fa3 is not None:
            y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
        else:
            y = torch.nn.functional.scaled_dot_product_attention(
                q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
                is_causal=True,
            ).transpose(1, 2)""", text
        )

        # Patch 3: bfloat16 → float16
        text = text.replace('torch.bfloat16', 'torch.float16')
        text = re.sub(
            r"autocast\(device_type=['\"]cuda['\"],\s*dtype=torch\.bfloat16\)",
            "autocast(device_type='cuda', dtype=torch.float16)", text
        )

        if text != original:
            train_path.write_text(text)
            print("[t4_patch] ✅ Done! Flash Attn 3 → SDPA, bf16 → fp16")
        else:
            print("[t4_patch] No changes needed (already patched)")
    else:
        print(f"[t4_patch] Ampere+ GPU detected — no patch needed")
else:
    print("[t4_patch] ⚠️ train.py not found")

In [15]:
# 資料準備（下載 + tokenizer，約 2 分鐘）
!uv run prepare.py --num-shards 10

Cache directory: /root/.cache/autoresearch

Data: all 11 shards already downloaded at /root/.cache/autoresearch/data

Tokenizer: already trained at /root/.cache/autoresearch/tokenizer

Done! Ready to train.


In [ ]:
# 跑一次訓練實驗（約 5 分鐘）
!uv run train.py 2>&1 | tee /tmp/run.log

# 提取結果
import re
log = open("/tmp/run.log").read()
bpb_match = re.findall(r"val_bpb[:\s]+([\d.]+)", log)
loss_match = re.findall(r"train_loss[:\s]+([\d.]+)", log)
step_match = re.findall(r"step[:\s]+(\d+)", log)

val_bpb = float(bpb_match[-1]) if bpb_match else None
train_loss = float(loss_match[-1]) if loss_match else None
steps = int(step_match[-1]) if step_match else None

print(f"\n{'='*40}")
print(f"val_bpb:    {val_bpb}")
print(f"train_loss: {train_loss}")
print(f"steps:      {steps}")
print(f"{'='*40}")

## 回傳結果到 MD.Piece API

設定你的 MD.Piece 後端 URL（如果是本機開發，可使用 ngrok 等工具暴露 port 8000）。

In [ ]:
# 回傳實驗結果到 MD.Piece（修改 API_URL 為你的後端位址）
import requests, json
from datetime import datetime

API_URL = "http://localhost:8000"  # @param {type:"string"}
EXPERIMENT_NAME = "baseline-t4"    # @param {type:"string"}
NOTES = "T4 baseline run"          # @param {type:"string"}

payload = {
    "name": EXPERIMENT_NAME,
    "val_bpb": val_bpb,
    "train_loss": train_loss,
    "steps": steps,
    "notes": NOTES,
    "kept": True,
}

try:
    r = requests.post(f"{API_URL}/research/", json=payload, timeout=10)
    if r.ok:
        print(f"✅ 已回傳到 MD.Piece: {r.json()}")
    else:
        print(f"⚠️ API 回傳錯誤 ({r.status_code}): {r.text}")
        print(f"   結果已存在本地，可稍後手動上傳")
except requests.exceptions.ConnectionError:
    print(f"⚠️ 無法連線到 {API_URL}")
    print(f"   請確認後端已啟動，或使用 ngrok 暴露 localhost:8000")
    print(f"\n📋 手動提交指令：")
    print(f'curl -X POST {API_URL}/research/ -H "Content-Type: application/json" -d \'{json.dumps(payload)}\'')

## 🔄 自動化實驗循環

下載 `autoresearch_runner.py` 並自動跑多輪實驗（假設 → 修改 → 訓練 → 評估 → keep/revert）。
每輪結果自動回傳到 MD.Piece API。

In [ ]:
# ── 自動化 Runner（直接嵌入，不需要從 GitHub 下載） ──
!pip install requests -q

# 寫入 runner 腳本
runner_code = r'''#!/usr/bin/env python3
import argparse, json, os, re, subprocess, sys, time
from datetime import datetime
from pathlib import Path

HYPOTHESES = [
    {"name": "deeper-model", "description": "n_layer 8->12", "patch": {"target": "n_layer", "old": "8", "new": "12"}},
    {"name": "wider-model", "description": "n_embd 512->768", "patch": {"target": "n_embd", "old": "512", "new": "768"}},
    {"name": "more-heads", "description": "n_head 4->8", "patch": {"target": "n_head", "old": "4", "new": "8"}},
    {"name": "larger-batch", "description": "TOTAL_BATCH_SIZE x2", "patch": {"target": "TOTAL_BATCH_SIZE", "find_multiply": 2}},
    {"name": "higher-lr", "description": "learning_rate x1.5", "patch": {"target": "learning_rate", "find_multiply": 1.5}},
    {"name": "lower-lr", "description": "learning_rate x0.5", "patch": {"target": "learning_rate", "find_multiply": 0.5}},
    {"name": "longer-warmup", "description": "warmup_steps x2", "patch": {"target": "warmup_steps", "find_multiply": 2}},
    {"name": "window-SSSS", "description": "window all sliding", "patch": {"target": "window_pattern", "old": "'SSSL'", "new": "'SSSS'"}},
    {"name": "window-SLSL", "description": "window alternating", "patch": {"target": "window_pattern", "old": "'SSSL'", "new": "'SLSL'"}},
    {"name": "sequence-4096", "description": "seq_len 2048->4096", "patch": {"target": "sequence_len", "old": "2048", "new": "4096"}},
]

TSV_HEADER = "name\tval_bpb\ttrain_loss\tsteps\tduration\tkept\tdescription\ttimestamp\n"

def init_tsv(p):
    if not p.exists(): p.write_text(TSV_HEADER)

def append_result(p, r):
    with open(p, "a") as f:
        f.write(f"{r['name']}\t{r.get('val_bpb','')}\t{r.get('train_loss','')}\t{r.get('steps','')}\t{r.get('duration_seconds','')}\t{r.get('kept','')}\t{r.get('description','')}\t{r.get('timestamp','')}\n")

def run_training(timeout_extra=60):
    print("  Training...", flush=True)
    start = time.time()
    try:
        result = subprocess.run(["uv", "run", "train.py"], capture_output=True, text=True, timeout=600+timeout_extra)
        log = result.stdout + result.stderr
    except subprocess.TimeoutExpired:
        return {"error": "Timed out", "duration_seconds": time.time()-start}
    duration = time.time() - start
    bpb = re.findall(r"val(?:_|\s)bpb[:\s]+([\d.]+)", log)
    loss = re.findall(r"train(?:_|\s)loss[:\s]+([\d.]+)", log)
    steps = re.findall(r"step[:\s]+(\d+)", log)
    val_bpb = float(bpb[-1]) if bpb else None
    if val_bpb is None and result.returncode != 0:
        return {"error": "Training failed", "duration_seconds": duration}
    return {"val_bpb": val_bpb, "train_loss": float(loss[-1]) if loss else None, "steps": int(steps[-1]) if steps else None, "duration_seconds": round(duration,1)}

def git_commit(msg):
    subprocess.run(["git", "add", "-A"], check=True)
    subprocess.run(["git", "commit", "-am", msg], check=True)

def git_revert():
    subprocess.run(["git", "reset", "--hard", "HEAD~1"], check=True)

def apply_patch(tp, patch):
    text = tp.read_text(); target = patch["target"]
    if "old" in patch and "new" in patch:
        if patch["old"] not in text: return False
        text = text.replace(patch["old"], patch["new"], 1)
    elif "find_multiply" in patch:
        m = re.search(rf"({target}\s*=\s*)([\d.]+)", text)
        if not m: return False
        nv = float(m.group(2)) * patch["find_multiply"]
        if nv == int(nv): nv = int(nv)
        text = text[:m.start()] + f"{m.group(1)}{nv}" + text[m.end():]
    else: return False
    tp.write_text(text); return True

def submit_api(url, r):
    if not url: return
    try:
        import requests
        requests.post(f"{url}/research/", json={"name":r["name"],"val_bpb":r.get("val_bpb"),"train_loss":r.get("train_loss"),"steps":r.get("steps"),"duration_seconds":r.get("duration_seconds"),"notes":r.get("description",""),"kept":r.get("kept",False)}, timeout=10)
    except: pass

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--rounds", type=int, default=5)
    p.add_argument("--api-url", type=str, default="")
    p.add_argument("--results-tsv", type=str, default="results.tsv")
    args = p.parse_args()
    tp = Path("train.py")
    if not tp.exists(): sys.exit("train.py not found")
    rp = Path(args.results_tsv); init_tsv(rp)
    print(f"AutoResearch Runner: {args.rounds} rounds")
    print("Round 0: Baseline")
    bl = run_training()
    if "error" in bl: sys.exit(f"Baseline failed: {bl['error']}")
    best = bl["val_bpb"]; print(f"  Baseline val_bpb: {best}")
    br = {"name":"baseline","description":"Initial baseline","kept":True,"timestamp":datetime.utcnow().isoformat(),**bl}
    append_result(rp, br); submit_api(args.api_url, br)
    for i, h in enumerate(HYPOTHESES[:args.rounds], 1):
        print(f"\nRound {i}/{args.rounds}: {h['name']} — {h['description']}")
        if not apply_patch(tp, h["patch"]): print("  Skip"); continue
        git_commit(f"hypothesis: {h['description']}")
        r = run_training()
        if "error" in r: kept=False; git_revert()
        elif r["val_bpb"] is not None and r["val_bpb"] < best: print(f"  KEPT val_bpb: {r['val_bpb']}"); best=r["val_bpb"]; kept=True
        else: print(f"  REVERTED val_bpb: {r.get('val_bpb')}"); git_revert(); kept=False
        er = {"name":h["name"],"description":h["description"],"kept":kept,"timestamp":datetime.utcnow().isoformat(),**{k:v for k,v in r.items() if k!="error"}}
        append_result(rp, er); submit_api(args.api_url, er)
    print(f"\nDone! Best val_bpb: {best}")

if __name__ == "__main__": main()
'''

Path("autoresearch_runner.py").write_text(runner_code)
print("✅ Runner script written")

# 設定參數
API_URL = "http://localhost:8000"  # @param {type:"string"}
ROUNDS = 5  # @param {type:"integer"}

# 初始化 git（runner 需要 git commit/revert）
!git init . 2>/dev/null; git add -A; git commit -m "initial" --allow-empty 2>/dev/null

# 開始自動實驗循環！
!python autoresearch_runner.py --rounds {ROUNDS} --api-url {API_URL}

In [ ]:
# 查看結果摘要
import pandas as pd
df = pd.read_csv("results.tsv", sep="\t")
print(df.to_string(index=False))
print(f"\n🏆 Best: {df.loc[df['val_bpb'].idxmin(), 'name']} — val_bpb: {df['val_bpb'].min()}")